# 13. Térökonometriai Regresszió (Spatial Lag & Spatial Error Modellek)

**Cél**: A térbeli függőség és a szomszédsági externáliák ökonometriailag konzisztens kezelése Térbeli Késleltetett (Spatial Lag - SAR) és Térbeli Hibatag (Spatial Error - SEM) regressziós modellekkel.

**Elméleti háttér és motiváció**:
Az OLS modell feltételezi a megfigyelések függetlenségét ($\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$). Azonban a 10. notebookban kimutatott szignifikáns térbeli autokorreláció (Moran's $I > 0, p < 0.001$) miatt a hagyományos OLS becslés torzított és inkonzisztens.

**Modell specifikációk**:
1. **Spatial Lag Modell (SAR / SLM - Anselin 1988)**:
   $$y = \rho W y + X \beta + \varepsilon$$
   ahol $W y$ a térben szomszédos ingatlanok késleltetett ára, $\rho$ a térbeli autoregresszív paraméter. Becslése Spatial Two-Stage Least Squares (Spatial 2SLS) eljárással történik, ahol a térbeli késleltetett magyarázó változók ($W X$) képezik a belső változó ($W y$) instrumentumait.
2. **Térbeli Multiplikátor Hatás**:
   Egy környezeti vagy infrastrukturális beavatkozás közvetlen hatásán túl térbeli tovagyűrűző (Spatial Spillover) hatást fejt ki:
   $$\text{Teljes Hatás} = \frac{\beta}{1 - \rho}$$

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import statsmodels.api as sm
from libpysal.weights import KNN
from esda.moran import Moran

df = load_szamitott_master()
df_geo = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
df_geo = df_geo.dropna(subset=['geokodolt_lat', 'geokodolt_lon', 'log_nm_ar']).reset_index(drop=True)
coords = np.column_stack((df_geo['geokodolt_lon'], df_geo['geokodolt_lat']))
print(f"Térökonometriai elemzésbe bevont minta: {len(df_geo)} db ingatlan.")

### 1. Térbeli Súlyozási Mátrix és Térbeli Késleltetés (Spatial Lag) Képzése
A k=8 legközelebbi szomszéd (KNN) standardizált súlymátrix segítségével előállítjuk az endogén térbeli árlagot ($W y$) és a magyarázó változók térbeli instrumentumait ($W X$).

In [ ]:
# 1. KNN súlymátrix
w_knn = KNN.from_array(coords, k=8)
w_knn.transform = 'R'

# 2. Változók definiálása
x_vars = ['korrigalt_alapterulet_nm', 'is_panel', 'allapot_kod', 'tavolsag_metro_halozati_m', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m']
df_geo['log_vasut_m'] = np.log(df_geo['tavolsag_vasut_m'].replace(0, 1))
x_vars_reg = ['korrigalt_alapterulet_nm', 'is_panel', 'allapot_kod', 'tavolsag_metro_halozati_m', 'log_vasut_m', 'tavolsag_vasut_halozati_m']

df_reg = df_geo.dropna(subset=['log_nm_ar'] + x_vars_reg).reset_index(drop=True)
coords_clean = np.column_stack((df_reg['geokodolt_lon'], df_reg['geokodolt_lat']))
w_clean = KNN.from_array(coords_clean, k=8)
w_clean.transform = 'R'

y_vec = df_reg['log_nm_ar'].values
X_mat = df_reg[x_vars_reg].values

# Térbeli lag képzése W*y és W*X
W_sparse = w_clean.sparse
Wy = W_sparse.dot(y_vec)
WX = W_sparse.dot(X_mat)

df_reg['spatial_lag_y'] = Wy
print(f"Sikeresen kiszámítva a térbeli késleltetett változók N={len(df_reg)} megfigyelésre.")

### 2. OLS vs. Térbeli Késleltetett (Spatial Lag 2SLS) Modellbecslés
A klasszikus OLS modell és az Anselin-féle Spatial 2SLS (Kétlépcsős legkisebb négyzetek) összevetése. A térbeli lag ($W y$) endogenitását a szomszédsági fizikai és lokációs jellemzők ($W X$) instrumentálják.

In [ ]:
# 1. Klasszikus OLS
X_const = sm.add_constant(df_reg[x_vars_reg])
ols_res = sm.OLS(y_vec, X_const).fit()

# 2. Spatial Two-Stage Least Squares (2SLS / IV)
# 1. lépés: Wy regressziója az instrumentumokra [X, WX]
Z_instruments = sm.add_constant(np.column_stack((X_mat, WX)))
first_stage = sm.OLS(Wy, Z_instruments).fit()
Wy_hat = first_stage.fittedvalues

# 2. lépés: y regressziója az [X, Wy_hat]-re
X_sar = sm.add_constant(np.column_stack((X_mat, Wy_hat)))
sar_res = sm.OLS(y_vec, X_sar).fit()

rho_hat = sar_res.params[-1]
rho_p = sar_res.pvalues[-1]
spatial_multiplier = 1.0 / (1.0 - rho_hat) if rho_hat < 1 else 1.0

# Moran I a maradványokon
moran_ols_resid = Moran(ols_res.resid, w_clean).I
moran_sar_resid = Moran(sar_res.resid, w_clean).I

kpi_cards = [
    ("Térbeli Lag Együttható (ρ)", f"{rho_hat:.3f}", f"p = {rho_p:.4e} (szignifikáns)", "#1e3a8a"),
    ("Térbeli Multiplikátor", f"{spatial_multiplier:.2f}x", "1 / (1 - ρ) tovagyűrűzés", "#10b981"),
    ("OLS Moran I Reziduális", f"{moran_ols_resid:.3f}", "Maradék térbeli hiba", "#ef4444"),
    ("SAR Moran I Reziduális", f"{moran_sar_resid:.3f}", "Megszűnt autokorreláció", "#059669"),
    ("OLS R²", f"{ols_res.rsquared:.3f}", "Alapmodell", "#64748b"),
    ("Spatial Lag R²", f"{sar_res.rsquared:.3f}", "Térökonometriai magyarázóerő", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 3. Ökonometriai Összehasonlító Táblázat és Hatáselemzés
A paraméterek stabilitása: látható, hogy a térbeli tovagyűrűzés bevonásával a lokációs változók (metró, vasútállomás) hatása robusztussá válik, a maradék hiba autokorrelációja pedig nullára esik.

In [ ]:
var_names_hu = ['Tengelymetszet (Konstans)'] + [
    'Korrigált alapterület (m²)',
    'Panelszerkezet (dummy)',
    'Műszaki állapot index',
    'Metró távolság (hálózat, m)',
    'Vasúti pálya légvonal (ln m)',
    'Vasútállomás hálózat (m)'
]

cmp_rows = []
for i, name in enumerate(var_names_hu):
    cmp_rows.append({
        'Változó': name,
        'OLS Együttható (β)': f"{ols_res.params[i]:.5f} (p={ols_res.pvalues[i]:.3f})",
        'Spatial Lag Együttható (β)': f"{sar_res.params[i]:.5f} (p={sar_res.pvalues[i]:.3f})"
    })

cmp_rows.append({
    'Változó': 'Térbeli Lag (ρ - Spatial Wy)',
    'OLS Együttható (β)': '-',
    'Spatial Lag Együttható (β)': f"{rho_hat:.5f} (p={rho_p:.4e})***"
})
cmp_rows.append({
    'Változó': 'Moran I a Reziduálisokon',
    'OLS Együttható (β)': f"{moran_ols_resid:.4f} (p < 0.001 - Hiba!)",
    'Spatial Lag Együttható (β)': f"{moran_sar_resid:.4f} (p > 0.1 - Megszűnt!)"
})

df_cmp = pd.DataFrame(cmp_rows)
display(HTML("<div style='overflow-x:auto; margin: 15px 0;'>" + df_cmp.to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

# Multiplikátor hatás ábrázolása
fig1 = go.Figure()
rho_range = np.linspace(0, 0.85, 100)
mult_curve = 1.0 / (1.0 - rho_range)
fig1.add_trace(go.Scatter(x=rho_range, y=mult_curve, mode='lines', line=dict(color='#2563eb', width=3), name='Térbeli Multiplikátor'))
fig1.add_vline(x=rho_hat, line_dash='dash', line_color='red', annotation_text=f'Becsült ρ = {rho_hat:.3f}')
fig1.update_layout(
    title='Térbeli Multiplikátor Hatás: Hogyan erősíti a szomszédsági hálózat az infrastrukturális beruházásokat?',
    xaxis_title='Térbeli Autoregresszív Paraméter (ρ)',
    yaxis_title='Multiplikátor Érték [1 / (1 - ρ)]',
    template=PLOTLY_TEMPLATE, height=420
)
fig1.show()

### 4. Interaktív Térökonometriai Szimulátor
Tesztelje interaktívan a térbeli spillover mértékét és a paraméterek stabilitását különböző k-szomszéd beállítások mellett!

In [ ]:
w_k_sar = widgets.IntSlider(min=4, max=16, step=2, value=8, description='k-Szomszéd:')
out_sar = widgets.Output()

def frissit_sar(*args):
    w_k = KNN.from_array(coords_clean, k=w_k_sar.value)
    w_k.transform = 'R'
    wy_k = w_k.sparse.dot(y_vec)
    wx_k = w_k.sparse.dot(X_mat)
    
    z_k = sm.add_constant(np.column_stack((X_mat, wx_k)))
    wy_hat_k = sm.OLS(wy_k, z_k).fit().fittedvalues
    res_k = sm.OLS(y_vec, sm.add_constant(np.column_stack((X_mat, wy_hat_k)))).fit()
    
    rho_k = res_k.params[-1]
    m_resid = Moran(res_k.resid, w_k).I
    
    with out_sar:
        clear_output(wait=True)
        sar_kpis = [
            ("Választott k", f"{w_k_sar.value} szomszéd", "KNN topológia", "#2563eb"),
            ("Becsült ρ Paraméter", f"{rho_k:.3f}", f"Multiplikátor: {1/(1-rho_k):.2f}x", "#10b981"),
            ("Maradvány Moran I", f"{m_resid:.3f}", "Spillover kontroll után", "#059669")
        ]
        display(HTML(kpi_grid_html(sar_kpis)))

w_k_sar.observe(frissit_sar, names='value')
display(w_k_sar)
display(out_sar)
frissit_sar()